In [1]:
# --- Reproducibility Seed Setup ---
import os
import random
import numpy as np
import torch

SEED = 3180

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Global random seed set:", SEED)

✅ Global random seed set: 3180


In [2]:
import sys, site, platform
print("PY:", platform.python_version())
print("EXE:", sys.executable)
print("USER_SITE:", site.getusersitepackages())

PY: 3.9.18
EXE: /common/software/install/manual/pytorch/2.1.2-pyclustertend/bin/python
USER_SITE: /users/4/volko028/.local/lib/python3.9/site-packages


In [3]:
import torch, transformers, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.1.2
transformers: 4.45.2
accelerate: 1.10.1


In [4]:
import os, torch, platform
print("python:", platform.python_version())
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("cuda.is_available():", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count(), "visible:", os.environ.get("CUDA_VISIBLE_DEVICES"))

python: 3.9.18
torch: 2.1.2 cuda: 11.8
cuda.is_available(): True
device_count: 1 visible: 0


In [5]:
# --- Imports ---
from sklearn.model_selection import train_test_split
import math, torch, pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
import numpy as np
from sklearn.metrics import roc_auc_score
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import classification_report

from sklearn.metrics import (
    precision_recall_curve, f1_score, precision_score, recall_score, roc_auc_score
)
from scipy.special import expit
import numpy as np

2026-03-09 16:01:05.188044: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-09 16:01:05.188143: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-09 16:01:05.188156: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-09 16:01:05.197252: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
# --- Config ---
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN    = 256
PER_DEVICE_BS = 2
GRAD_ACCUM    = 4
NUM_EPOCHS    = 3
EVAL_EVERY    = 2000
SAVE_EVERY    = 2000
STRIDE=32

In [15]:
# --- Load data ---
import pandas as pd

use_cols = ["notes", "outcome_critical"]
train_df = pd.read_csv("mv_train_DISPOSITION.csv", usecols = use_cols)
val_df   = pd.read_csv("mv_val_DISPOSITION.csv", usecols = use_cols)
test_df  = pd.read_csv("mv_test_DISPOSITION.csv", usecols = use_cols)

In [16]:
train_df['outcome_critical'].value_counts()

False    135859
True      16032
Name: outcome_critical, dtype: int64

In [17]:
# --- Tokenizer & collator ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

from transformers import DataCollatorWithPadding
import torch

data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

def tokenize_with_stride(text: str):
    text = text[:6000]
    return tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        padding=False
    )

In [18]:
# --- Streaming Dataset Loader ---
import math
import pandas as pd
import torch
import random
import numpy as np

class StreamingNotes(torch.utils.data.IterableDataset):
    def __init__(
        self,
        csv_path: str,
        dup_pos: int = 1,
        shuffle_within_chunk: bool = True,
        seed: int = 0,
        tokenizer=None,
        length_cache=None,
        max_len: int = 256,
        stride: int = 32,
        max_char: int = 6000,
        chunksize: int = 8192,
        text_col: str = "notes",
        label_col: str = "outcome_critical",
    ):
        super().__init__()
        self.csv_path = csv_path
        self.dup_pos = max(1, int(dup_pos))
        self.shuffle_within_chunk = shuffle_within_chunk
        self.seed = seed

        self.tokenizer = tokenizer
        self.max_len = int(max_len)
        self.stride = int(stride)
        self.max_char = int(max_char)
        self.chunksize = int(chunksize)

        self.text_col = text_col
        self.label_col = label_col

        self._length = length_cache
        if self._length is None and self.tokenizer is not None:
            self._length = self._estimate_length()

    def __len__(self):
        if self._length is None:
            raise TypeError(
                "StreamingNotes has no length. Pass tokenizer=... so it can estimate __len__, "
                "or pass length_cache=<int>."
            )
        return int(self._length)

    def _tokenize_batch(self, texts):
        return self.tokenizer(
            texts,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_len,
            stride=self.stride,
            return_overflowing_tokens=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            padding=False
        )

    def _estimate_length(self):
        total = 0
        usecols = [self.text_col, self.label_col]

        for chunk in pd.read_csv(self.csv_path, chunksize=self.chunksize, usecols=usecols):
            chunk = chunk.dropna(subset=usecols)
            if chunk.empty:
                continue

            texts = chunk[self.text_col].astype(str).str.slice(0, self.max_char).tolist()
            labels = chunk[self.label_col].astype(int).to_numpy()

            enc = self._tokenize_batch(texts)
            mapping = enc.get("overflow_to_sample_mapping", None)

            if mapping is None:
                for y in labels:
                    repeats = self.dup_pos if y == 1 else 1
                    total += repeats
            else:
                mapping = np.asarray(mapping)
                for j in range(len(texts)):
                    n_win = int(np.sum(mapping == j))
                    repeats = self.dup_pos if labels[j] == 1 else 1
                    total += repeats * n_win

        return int(total)

    def __iter__(self):
        rng = random.Random(self.seed)
        usecols = [self.text_col, self.label_col]

        global_note_id = 0

        for chunk in pd.read_csv(self.csv_path, chunksize=self.chunksize, usecols=usecols):
            chunk = chunk.dropna(subset=usecols)
            if chunk.empty:
                continue

            rows = list(chunk.iterrows())
            if self.shuffle_within_chunk:
                rng.shuffle(rows)

            for _, row in rows:
                y = int(row[self.label_col])
                text = str(row[self.text_col])[:self.max_char]

                enc = self.tokenizer(
                    text,
                    add_special_tokens=True,
                    truncation=True,
                    max_length=self.max_len,
                    stride=self.stride,
                    return_overflowing_tokens=True,
                    return_attention_mask=True,
                    return_token_type_ids=True,
                    padding=False
                )

                n = len(enc["input_ids"])
                repeats = self.dup_pos if y == 1 else 1

                for _ in range(repeats):
                    for k in range(n):
                        item = {
                            "input_ids": enc["input_ids"][k],
                            "attention_mask": enc["attention_mask"][k],
                            "labels": y,
                            "note_id": global_note_id,
                        }
                        if "token_type_ids" in enc:
                            item["token_type_ids"] = enc["token_type_ids"][k]
                        yield item

                global_note_id += 1
                
                
                
train_ds = StreamingNotes(
    "mv_train_DISPOSITION.csv",
    dup_pos=3,
    shuffle_within_chunk=True,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)

val_ds = StreamingNotes(
    "mv_val_DISPOSITION.csv",
    dup_pos=1,
    shuffle_within_chunk=False,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)


In [19]:
# --- Model ---
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.id2label = {0: "NoCritical", 1: "Critical"}
model.config.label2id = {"NoCritical": 0, "Critical": 1}

/common/software/install/manual/pytorch/2.1.2-pyclustertend/lib/python3.9/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
# --- Training Arguments ---
n_train = len(train_ds)
world_size = max(1, torch.cuda.device_count())
effective_bsz = PER_DEVICE_BS * GRAD_ACCUM * world_size
steps_per_epoch = math.ceil(n_train / effective_bsz)
MAX_STEPS = steps_per_epoch * NUM_EPOCHS
print(f"Training rows={n_train}  steps/epoch={steps_per_epoch}  max_steps={MAX_STEPS}")
assert MAX_STEPS > 0, "No training rows found. Check CSV path and columns."

training_args = TrainingArguments(
    output_dir="runs/clin-longformer",
    seed=SEED,
    data_seed=SEED,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    tf32=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    eval_accumulation_steps=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    save_strategy="steps",
    save_steps=SAVE_EVERY,
    load_best_model_at_end=True,
    metric_for_best_model="note_auroc",
    greater_is_better=True,
    save_total_limit=3,
    remove_unused_columns=False,
    include_inputs_for_metrics=True,
    report_to=["none"],
    gradient_checkpointing=True,
)

Training rows=185098  steps/epoch=23138  max_steps=69414


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
# --- Create Trainer ---
import numpy as np
import torch
from transformers import Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score

class NoteAwareTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs)
        inputs.pop("note_id", None)
        return super().compute_loss(model, inputs, return_outputs=return_outputs)

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = dict(inputs)
        note_ids = inputs.pop("note_id", None)
        loss, logits, labels = super().prediction_step(
            model, inputs, prediction_loss_only, ignore_keys
        )

        if note_ids is not None:
            if not isinstance(note_ids, torch.Tensor):
                note_ids = torch.as_tensor(note_ids, dtype=torch.long)
            note_ids = note_ids.view(-1, 1)
            return loss, (logits, note_ids), labels

        return loss, logits, labels

POOL = "mean"

def compute_metrics_note_level(eval_pred):
    preds_obj, labels = eval_pred.predictions, eval_pred.label_ids
    logits, note_ids = preds_obj
    if isinstance(logits, torch.Tensor): logits = logits.detach().cpu().numpy()
    if isinstance(note_ids, torch.Tensor): note_ids = note_ids.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor): labels = labels.detach().cpu().numpy()
    note_ids = note_ids.squeeze(-1)

    logits = np.asarray(logits)
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)
    p1 = probs[:, 1]

    # pool by note
    by_note_probs, by_note_label = {}, {}
    for nid, pr, y in zip(note_ids, p1, labels):
        nid = int(nid)
        by_note_probs.setdefault(nid, []).append(float(pr))
        by_note_label[nid] = int(y)

    if POOL == "max":
        note_probs = np.array([max(v) for nid, v in sorted(by_note_probs.items())])
    else:
        note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
    note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

    # rank metrics
    try:
        auroc = roc_auc_score(note_labels, note_probs)
    except Exception:
        auroc = float("nan")
    try:
        auprc = average_precision_score(note_labels, note_probs)
    except Exception:
        auprc = float("nan")

    # find best F1 threshold on this eval set
    precs, recs, ths = precision_recall_curve(note_labels, note_probs)
    f1s = 2 * precs * recs / (precs + recs + 1e-12)
    best_ix = int(np.nanargmax(f1s))
    best_th = ths[best_ix-1] if best_ix > 0 and best_ix <= len(ths) else 0.5
    target_p = 0.80
    ix_p = np.where(precs >= target_p)[0]

    # report metrics at best_th
    note_pred_best = (note_probs >= best_th).astype(int)
    acc = accuracy_score(note_labels, note_pred_best)
    prec, rec, f1, _ = precision_recall_fscore_support(note_labels, note_pred_best, average="binary", zero_division=0)
    
    print({
      "true_pos_rate": float(np.mean(note_labels)),
      "pred_pos_rate@0.5": float(np.mean((note_probs >= 0.5).astype(int))),
      "mean_prob_pos": float(np.mean(note_probs[note_labels==1])) if np.any(note_labels==1) else None,
      "mean_prob_neg": float(np.mean(note_probs[note_labels==0])) if np.any(note_labels==0) else None,
      "best_th": float(best_th),
    })

    return {
        "note_auroc": auroc,
        "note_auprc": auprc,
        "note_f1": f1,
        "note_precision": prec,
        "note_recall": rec,
        "note_acc": acc,
        "th_best_f1": float(best_th),
    }

In [23]:
# --- Model Training ---
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from transformers import TrainerCallback
import time

trainer = NoteAwareTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_note_level,
    )
trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss,Validation Loss,Note Auroc,Note Auprc,Note F1,Note Precision,Note Recall,Note Acc,Th Best F1
2000,0.567200,0.331341,0.699561,0.224870,0.298773,0.229822,0.426829,0.792321,0.238462
4000,0.532700,0.295610,0.723887,0.338316,0.339965,0.400690,0.295224,0.881176,0.180974
6000,0.515000,0.364175,0.775749,0.391248,0.387256,0.421399,0.358232,0.882492,0.586419
8000,0.535900,0.276473,0.782322,0.423682,0.397590,0.365417,0.435976,0.863057,0.251096
10000,0.549600,0.295391,0.790099,0.430043,0.406688,0.393997,0.420224,0.872906,0.384912
12000,0.539100,0.279955,0.786978,0.422870,0.398898,0.435620,0.367886,0.885073,0.273769
14000,0.554600,0.291990,0.786009,0.424999,0.397207,0.404423,0.390244,0.877225,0.326218
16000,0.496300,0.317650,0.790567,0.430985,0.399296,0.395222,0.403455,0.874170,0.456531
18000,0.454700,0.277950,0.788255,0.421823,0.388453,0.356054,0.427337,0.860529,0.251786
20000,0.592900,0.280113,0.796849,0.447215,0.415060,0.374846,0.464939,0.864163,0.266548


{'true_pos_rate': 0.10365532497629833, 'pred_pos_rate@0.5': 0.007847887917412831, 'mean_prob_pos': 0.24239439424884512, 'mean_prob_neg': 0.16895214571125122, 'best_th': 0.23846150934696198}
{'true_pos_rate': 0.10365532497629833, 'pred_pos_rate@0.5': 0.012693563678499947, 'mean_prob_pos': 0.2043097110851327, 'mean_prob_neg': 0.08868607633437084, 'best_th': 0.18097351491451263}
{'true_pos_rate': 0.10365532497629833, 'pred_pos_rate@0.5': 0.11777098914989993, 'mean_prob_pos': 0.4705585737890617, 'mean_prob_neg': 0.212505943844775, 'best_th': 0.5864192843437195}
{'true_pos_rate': 0.10365532497629833, 'pred_pos_rate@0.5': 0.04361108184978405, 'mean_prob_pos': 0.3342663576633374, 'mean_prob_neg': 0.11030571319971473, 'best_th': 0.25109636783599854}
{'true_pos_rate': 0.10365532497629833, 'pred_pos_rate@0.5': 0.06199304750869061, 'mean_prob_pos': 0.4027355007232901, 'mean_prob_neg': 0.1465858565187017, 'best_th': 0.3849121630191803}
{'true_pos_rate': 0.10365532497629833, 'pred_pos_rate@0.5': 0.

TrainOutput(global_step=69414, training_loss=0.5246066413342148, metrics={'train_runtime': 23768.676, 'train_samples_per_second': 23.363, 'train_steps_per_second': 2.92, 'total_flos': 5.890906062875616e+16, 'train_loss': 0.5246066413342148, 'epoch': 3.000097245783315})

In [24]:
test_ds = StreamingNotes(
    "mv_test_DISPOSITION.csv",
    dup_pos=1,
    shuffle_within_chunk=False,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)

In [25]:
import sys
import math
import torch
import numpy as np
from tqdm import tqdm
from transformers import EvalPrediction
from sklearn.metrics import confusion_matrix, classification_report

model = trainer.model
model.eval()

eval_loader = trainer.get_eval_dataloader(test_ds)

n_val = len(test_ds)
world_size = max(1, torch.cuda.device_count())
bsz = trainer.args.per_device_eval_batch_size * world_size

total_steps = max(1, math.ceil(n_val / bsz))
print(f"n_val={n_val}, bsz={bsz}, total_steps={total_steps}", flush=True)

all_logits = []
all_labels = []
all_note_ids = []

eval_iter = iter(eval_loader)

for _ in tqdm(
    range(total_steps),
    desc="Predicting",
    file=sys.stdout,
    disable=False,
    ascii=True,
    mininterval=1.0,
):
    try:
        batch = next(eval_iter)
    except StopIteration:
        break

    batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
    note_ids = batch.pop("note_id")
    labels   = batch["labels"]

    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits

    all_logits.append(logits.cpu().numpy())
    all_labels.append(labels.cpu().numpy())
    all_note_ids.append(note_ids.cpu().numpy())

logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)
note_ids = np.concatenate(all_note_ids, axis=0).reshape(-1, 1)

eval_pred = EvalPrediction(
    predictions=(logits, note_ids),
    label_ids=labels,
)
metrics = compute_metrics_note_level(eval_pred)
print("\nValidation metrics:", metrics)

# rebuild pooled note probabilities
logits_np = logits
probs = np.exp(logits_np - logits_np.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
p1 = probs[:, 1]

by_note_probs, by_note_label = {}, {}
for nid, pr, y in zip(note_ids.squeeze(-1), p1, labels):
    nid = int(nid)
    by_note_probs.setdefault(nid, []).append(float(pr))
    by_note_label[nid] = int(y)

note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

best_th = metrics["th_best_f1"]
note_pred = (note_probs >= best_th).astype(int)

print("\nConfusion matrix:")
print(confusion_matrix(note_labels, note_pred))

print("\nClassification report:")
print(classification_report(note_labels, note_pred, digits=4))


n_val=19084, bsz=2, total_steps=9542
Predicting: 100%|##########| 9542/9542 [01:39<00:00, 96.33it/s]
{'true_pos_rate': 0.10417654184441986, 'pred_pos_rate@0.5': 0.09348501606362247, 'mean_prob_pos': 0.42583043762533085, 'mean_prob_neg': 0.10195880650946738, 'best_th': 0.43116500973701477}

Validation metrics: {'note_auroc': 0.8072688693150673, 'note_auprc': 0.44115074302335044, 'note_f1': 0.40641711229946526, 'note_precision': 0.409440738840431, 'note_recall': 0.4034378159757331, 'note_acc': 0.8772317901722231, 'th_best_f1': 0.43116500973701477}

Confusion matrix:
[[15858  1151]
 [ 1180   798]]

Classification report:
              precision    recall  f1-score   support

           0     0.9307    0.9323    0.9315     17009
           1     0.4094    0.4034    0.4064      1978

    accuracy                         0.8772     18987
   macro avg     0.6701    0.6679    0.6690     18987
weighted avg     0.8764    0.8772    0.8768     18987



In [26]:
import pandas as pd

note_ids_sorted = np.array(sorted(by_note_label.keys()))

pred_df = pd.DataFrame({
    "sample_id": note_ids_sorted,
    "y_true": note_labels,
    "pred_prob_notes": note_probs
})

print(pred_df.head())

   sample_id  y_true  pred_prob_notes
0          0       0         0.121842
1          1       0         0.005888
2          2       0         0.005323
3          3       0         0.045820
4          4       0         0.002736


In [27]:
pred_df

,sample_id,y_true,pred_prob_notes
0,0,0,0.121842
1,1,0,0.005888
2,2,0,0.005323
3,3,0,0.045820
4,4,0,0.002736
...,...,...,...
18982,18982,1,0.003629
18983,18983,0,0.039937
18984,18984,0,0.961425
18985,18985,0,0.201931


In [28]:
from sklearn.metrics import roc_auc_score

auroc = roc_auc_score(note_labels, note_probs)
print("Note-level AUROC:", auroc)

Note-level AUROC: 0.8072688693150673


In [29]:
pred_df.to_csv("notes_test_predictions_task4.csv", index=False)

In [30]:
import sys
import math
import torch
import numpy as np
from tqdm import tqdm
from transformers import EvalPrediction
from sklearn.metrics import confusion_matrix, classification_report

model = trainer.model
model.eval()

# use validation dataset
eval_loader = trainer.get_eval_dataloader(val_ds)

# number of validation examples
n_test = len(val_ds)

world_size = max(1, torch.cuda.device_count())
bsz = trainer.args.per_device_eval_batch_size * world_size

total_steps = max(1, math.ceil(n_test / bsz))
print(f"n_test={n_test}, bsz={bsz}, total_steps={total_steps}", flush=True)

all_logits = []
all_labels = []
all_note_ids = []

eval_iter = iter(eval_loader)

for _ in tqdm(
    range(total_steps),
    desc="Predicting",
    file=sys.stdout,
    disable=False,
    ascii=True,
    mininterval=1.0,
):
    try:
        batch = next(eval_iter)
    except StopIteration:
        break

    batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
    note_ids = batch.pop("note_id")
    labels   = batch["labels"]

    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits

    all_logits.append(logits.cpu().numpy())
    all_labels.append(labels.cpu().numpy())
    all_note_ids.append(note_ids.cpu().numpy())

# concatenate predictions
logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)
note_ids = np.concatenate(all_note_ids, axis=0).reshape(-1, 1)

eval_pred = EvalPrediction(
    predictions=(logits, note_ids),
    label_ids=labels,
)

metrics = compute_metrics_note_level(eval_pred)
print("\nTest metrics:", metrics)

# rebuild pooled note probabilities
logits_np = logits
probs = np.exp(logits_np - logits_np.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
p1 = probs[:, 1]

by_note_probs, by_note_label = {}, {}
for nid, pr, y in zip(note_ids.squeeze(-1), p1, labels):
    nid = int(nid)
    by_note_probs.setdefault(nid, []).append(float(pr))
    by_note_label[nid] = int(y)

note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

best_th = metrics["th_best_f1"]
note_pred = (note_probs >= best_th).astype(int)

print("\nConfusion matrix:")
print(confusion_matrix(note_labels, note_pred))

print("\nClassification report:")
print(classification_report(note_labels, note_pred, digits=4))

n_test=19095, bsz=2, total_steps=9548
Predicting: 100%|##########| 9548/9548 [01:39<00:00, 96.15it/s]
{'true_pos_rate': 0.10365532497629833, 'pred_pos_rate@0.5': 0.09675550405561993, 'mean_prob_pos': 0.44055261274473645, 'mean_prob_neg': 0.10136480989799412, 'best_th': 0.48691093921661377}

Test metrics: {'note_auroc': 0.8023329942614563, 'note_auprc': 0.4561096358605684, 'note_f1': 0.42255717255717257, 'note_precision': 0.4324468085106383, 'note_recall': 0.41310975609756095, 'note_acc': 0.8829663962920047, 'th_best_f1': 0.48691093921661377}

Confusion matrix:
[[15951  1067]
 [ 1155   813]]

Classification report:
              precision    recall  f1-score   support

           0     0.9325    0.9373    0.9349     17018
           1     0.4324    0.4131    0.4226      1968

    accuracy                         0.8830     18986
   macro avg     0.6825    0.6752    0.6787     18986
weighted avg     0.8806    0.8830    0.8818     18986



In [31]:
import pandas as pd

note_ids_sorted = np.array(sorted(by_note_label.keys()))

pred_df = pd.DataFrame({
    "sample_id": note_ids_sorted,
    "y_true": note_labels,
    "pred_prob_notes": note_probs
})

print(pred_df.head())

   sample_id  y_true  pred_prob_notes
0          0       0         0.040617
1          1       0         0.739516
2          2       0         0.170715
3          3       1         0.999067
4          4       0         0.017127


In [32]:
pred_df.to_csv("notes_val_predictions_task4.csv", index=False)